In [1]:
# # скачиваем уже обработанную версию ямбды 500m, в которой только лайки

# !pip install -q gdown
# !gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
# !unzip -q dataset.zip

In [2]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=a3bc932b-42aa-4ace-a2de-2b06ec6117a2
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:04<00:00, 73.3MB/s]


### 1. Подготовка данных

**Задача:**
1) Считать данные (взаимодействия, эмбеддинги, метаданные).  
2) Оставить только взаимодействия, для которых есть эмбеддинги.  
3) Сделать core фильтрацию: оставить только айтемы с ≥5 взаимодействиями (в ДЗ мы делаем это исключительно для удобства и скорости, в реальной работе так делать не стоит)
4) Поджоинить метаданные (артисты треков) ко всем взаимодействиям. Ремарка: здесь у нас если у одного трека несколько артистов произойдет дублирование прослушивание - в рамках ДЗ мы ничего с этим не делаем, но в реальной жизни такого допускать нельзя. 
5) Сделать train-test split: последнюю неделю положить в тест.  
6) Ограничить тест юзерами, у которых есть взаимодействия в трейне.  
7) Подготовить для оценки качества `test_targets: Dict[uid, List[item_id]]`.
8) Оставить только эмбеддинги для core айтемов (остальные пофильтровать).

После этого блока должны существовать `train`, `test`, `embeddings`, `artists`, `test_targets`.

Для этого блока полезны как минимум следующие методы:
* `pl.read_parquet` - для чтения данных
* `.filter, .value_counts` помогут сделать core-фильтрацию
* `df.join(...)` - при джойне метаданных надо использовать `how='left'`, а не `how='inner'`
* `df.join(other, on=some_key, how='semi')` - режим `semi` используется для фильтраций (оставить только те строки из исходного датафрейма, ключ которых присутствует во второй таблице)

In [3]:
from typing import Dict, List
from collections import defaultdict
import os

import numpy as np
import polars as pl


DATA_DIR = "."
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60
TEMPORAL_THRESHOLD = 25_395_195

np.random.seed(42)

def _first_existing(candidates: List[str]) -> str | None:
    for name in candidates:
        path = os.path.join(DATA_DIR, name)
        if os.path.exists(path):
            return path
    return None


PATH_INTERACTIONS = _first_existing(["interactions.parquet", "likes.parquet", "listens.parquet",])

PATH_EMBEDDINGS = _first_existing([
    "embeddings.parquet",
    "item_embeddings.parquet",
    "track_embeddings.parquet",
])

interactions = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)

interactions = interactions.join(
    embeddings.select("item_id").unique(),
    on="item_id",
    how="semi",
)

core_items = (
    interactions
    .group_by("item_id")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM)
    .select("item_id")
)

data = interactions.join(core_items, on="item_id", how="semi")

train = data.filter(pl.col("timestamp") < TEMPORAL_THRESHOLD)
test = data.filter(pl.col("timestamp") >= TEMPORAL_THRESHOLD)
train_users = train.select("uid").unique()
test = test.join(train_users, on="uid", how="semi")

test_targets_df = (
    test
    .select(["uid", "item_id"])
    .unique(subset=["uid", "item_id"], maintain_order=True)
    .group_by("uid", maintain_order=True)
    .agg(pl.col("item_id"))
)
test_targets = dict(test_targets_df.iter_rows())

In [4]:
%%writefile tests_lib.py
from __future__ import annotations
from typing import Dict, List, Callable, Any

import numpy as np
import polars as pl


class Constants:
    temporal_threshold: int = 25395195
    train_size: int = 7755722
    test_size: int = 254164
    num_test_users: int = 37446


def check_data_split(
    train: pl.DataFrame,
    test: pl.DataFrame,
    test_targets: Dict[int, List[int]],
) -> None:
    # required columns
    for col in ["uid", "item_id", "timestamp"]:
        assert col in train.columns, f"train missing column {col}"
        assert col in test.columns, f"test missing column {col}"

    # test users in train
    train_users = set(train["uid"].unique().to_list())
    test_users = set(test["uid"].unique().to_list())
    assert test_users.issubset(train_users), "test contains users absent in train"

    # targets cover test users
    assert len(test_targets) > 0, "test_targets is empty"
    assert set(test_targets.keys()) == test_users, "test_targets keys must match test users"

    # items in train have embeddings (after filtering)
    train_items = set(train["item_id"].unique().to_list())

    # timestamp ordering (train < test threshold) – soft check
    assert train["timestamp"].max() <= test["timestamp"].max(), "Unexpected timestamps"

    assert train["timestamp"].max() < Constants.temporal_threshold, "Train-test temporal split is incorrect"
    assert train.height == Constants.train_size, "Train size is incorrect"
    assert test.height == Constants.test_size, "Test size is incorrect"
    assert len(test_targets) == Constants.num_test_users, "Test targets size is incorrect"
    print('All good! :)')


def check_metrics(
    get_metrics: Callable[[List[int], List[int], int], Dict[str, float]],
    evaluate: Callable[[Dict[int, List[int]], Dict[int, List[int]], int, int], Dict[str, float]],
) -> None:
    # toy example
    targets = [1, 2, 3]
    cands = [10, 2, 30, 1]
    m = get_metrics(targets, cands, topk=4)
    assert 0.0 <= m["hitrate"] <= 1.0
    assert 0.0 <= m["recall"] <= 1.0
    assert 0.0 <= m["ndcg"] <= 1.0
    assert np.isclose(m["hitrate"], 1.0) and np.isclose(m["recall"], 2. / 3) and np.isclose(m["ndcg"], 0.49818925746641285), \
    "'get_metrics' outputs incorrect values"

    targets_by_user = {7: [1, 2], 8: [9]}
    cands_by_user = {7: [1, 3, 4], 8: [10, 9, 11]}
    out = evaluate(targets_by_user, cands_by_user, catalog_size=100, topk=3)
    for k in ["hitrate", "recall", "ndcg", "coverage"]:
        assert k in out, f"missing metric {k}"
    assert (
        np.isclose(out["hitrate"], 1.0) 
        and np.isclose(out["recall"], 0.75) 
        and np.isclose(out["ndcg"], 0.622038473168458) 
        and np.isclose(out["coverage"], 0.06)
     ), "'evaluate' outputs incorrect values"
    print('All good! :)')
    

def test_tokenizer(item_to_token_id):
    assert 'token_id' in item_to_token_id.columns
    assert isinstance(item_to_token_id, pl.DataFrame)
    assert item_to_token_id.height == 30000
    assert (item_to_token_id.head(5)['item_id'].to_numpy() == np.array([5862961, 9378983, 6901374, 3542184, 5463340], dtype=np.uint32)).all()
    assert item_to_token_id['token_id'].max() == 30000
    assert item_to_token_id['token_id'].min() == 1


def test_train_data(train_events, train_histories):
    assert train_events.height == 3532362
    assert train_histories.height == 79307
    assert train_histories['token_id'].list.first().max() == 0
    assert train_histories['token_id'].list.len().max() == 100
    assert train_histories['token_id'].list.len().min() == 2
    
    assert np.isclose(train_histories['token_id'].list.len().mean(), 45.54035583239815)


Writing tests_lib.py


In [5]:
import tests_lib
tests_lib.check_data_split(train=train, test=test, test_targets=test_targets)

All good! :)


### 2. Оценка качества

#### 2.1 Определения метрик

2.1.1 Пусть для пользователя $u$:

* $G_u \subset \mathcal{I}$ — множество релевантных айтемов (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — упорядоченный список рекомендаций длины $K$

Обозначим индикатор релевантности $I_{u,k} = [ r_{u, k} \in G_u]$. В простонародье его еще часто называют `hits`.

2.1.2 **Hitrate@K** равен единичке, если мы угадали в topK хотя бы один релевантный айтем:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

2.1.3 **Recall@K** оценивает долю угаданных релевантных айтемов (от всех релевантных айтемов):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

2.1.4 Для подсчета **nDCG@K** нужно сначала посчитать **DCG@K**, затем посчитать **iDCG@K** (DCG в случае идеального ранжирования), затем одно поделить на другое:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

2.1.5 **Coverage@K** - это число уникальных айтемов во всех рекомендациях, деленное на размер каталога:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ где $\mathcal{I}_{train}$ — каталог айтемов в train.
* в качестве размера каталога используем количество айтемов, которые нам доступны для рекомендации на момент рекомендации (то есть количество уникальных айтемов в `train`)

#### 2.2 Что нужно сделать
Реализуйте функции:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Важно:** 
* `candidates[uid]` должен иметь длину ровно `topk`
* при подсчете метрики нужно дедуплицировать позитивы; это влияет на значение, на которое мы делим


In [6]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    candidates = candidates[:topk]

    target_set = set(targets)
    denom = min(len(target_set), topk)

    hits = np.array([1.0 if item in target_set else 0.0 for item in candidates], dtype=np.float64)

    hitrate = float(hits.sum() > 0)
    recall = float(hits.sum() / denom)

    discounts = 1.0 / np.log2(np.arange(2, topk + 2, dtype=np.float64))
    dcg = (hits * discounts).sum()
    idcg = discounts[:denom].sum()
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:

    metrics_sum = defaultdict(float)
    recommended_items = set()

    for uid, user_targets in targets.items():
        user_candidates = candidates[uid][:topk]
        
        m = get_metrics(user_targets, user_candidates, topk=topk)
        for name, value in m.items():
            metrics_sum[name] += value
        recommended_items.update(user_candidates)

    num_users = len(targets)
    out = {name: value / num_users for name, value in metrics_sum.items()}
    out["coverage"] = len(recommended_items) / catalog_size
    return out

In [7]:
tests_lib.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


### 3. Токенизация

Нужно построить таблицу item_to_token_id.

#### Что делаем

1. Берём `train['item_id']`
2. Считаем популярности `(value_counts)`
3. Сортируем:
* `count` ↓
* `item_id` ↑ (для детерминизма)
4. Берём `head(VOCAB_ITEMS)`
5. Назначаем `token_id = row_index + 1`
(0 зарезервирован под padding)
6. Оставляем только `['item_id', 'token_id']`

In [8]:
VOCAB_ITEMS = 30_000

item_to_token_id = (
    train
    .group_by("item_id")
    .agg(pl.len().alias("count"))
    .sort(by=["count", "item_id"], descending=[True, False])
    .head(VOCAB_ITEMS)
    .with_row_index(name="token_id")
    .with_columns([
        (pl.col("token_id") + 1).cast(pl.Int64).alias("token_id"),
    ])
    .select(["item_id", "token_id"])
)

In [9]:
tests_lib.test_tokenizer(item_to_token_id)

### 4. Обработка историй пользователя

Готовим последовательности для обучения.

#### Что нужно сделать

1. Оставить только те события, где `item_id` есть в `item_to_token_id`
   → `inner join`

2. Отсортировать по `timestamp`

3. Для каждого `uid` оставить последние `MAX_EVENTS_PER_USER` событий
   → `.group_by(...).tail(...)`

4. Построить `train_histories`:
   для каждого пользователя список `token_id`,
   где в начале добавлен `BOS = 0`


In [10]:
MAX_EVENTS_PER_USER = 100

train_events = train.join(item_to_token_id, on="item_id", how="inner").sort("timestamp").with_columns([pl.col("token_id").cast(pl.Int64)])
train_events = train_events.group_by("uid", maintain_order=True).tail(MAX_EVENTS_PER_USER - 1)
train_histories = pl.concat([
        train_events
        .select("uid")
        .unique(maintain_order=True)
        .with_columns(pl.lit(0, dtype=pl.Int64).alias("token_id")),

        train_events.select([
            pl.col("uid"),
            pl.col("token_id").cast(pl.Int64),
        ]),
    ]).group_by("uid", maintain_order=True).agg(pl.col("token_id"))

In [11]:
tests_lib.test_train_data(train_events, train_histories)

### 5. Датасет для обучения

Идея: мы превращаем таблицу с историями (`uid -> list[token_id]`) в поток **непрерывных** токенов и режем его на батчи длиной `batch_size * seq_len + 1`.

#### Что важно понять

* `__iter__` — это генератор: он **возвращает батчи через `yield`**.
* В начале каждой эпохи (каждого `__iter__`) можно перемешать пользователей:
  `df.sample(fraction=1.0, shuffle=True, seed=self.seed)`
* Паддинг **не делаем**: просто берём подряд идущие токены из потока.
* В один батч могут попасть куски разных пользователей — это ок, потому что **BOS=0** разделяет истории.
* На один батч нужно `batch_size * seq_len + 1` токен:

  * `inputs` — первые `batch_size * seq_len`
  * `targets` — те же токены, но сдвинутые на 1
  * `size = batch_size * seq_len`


In [12]:
from dataclasses import dataclass
import numpy as np
import polars as pl
import torch


@dataclass
class TrainingBatch:
    inputs: torch.Tensor
    targets: torch.Tensor
    size: int


class TrainingDataset:
    def __init__(
            self,
            df: pl.DataFrame,
            batch_size: int,
            seq_len: int,
            device: str = "cuda",
            chunk_rows: int = 64_000,
            shuffle: bool = True,
            seed: int | None = 42,
            pin_memory: bool = True
    ):
        self.df = df
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.device = device
        self.chunk_rows = chunk_rows
        self.shuffle = shuffle
        self.seed = seed
        self.pin_memory = pin_memory

        self.batch_num_tokens = batch_size * seq_len + 1
        self.total_num_tokens = int(self.df.get_column("token_id").list.len().sum())

    def __len__(self):
        return self.total_num_tokens // self.batch_num_tokens

    def __iter__(self):
        df = self.df
        if self.shuffle:
            df = df.sample(fraction=1.0, shuffle=True, seed=self.seed)

        buf = np.empty(self.batch_num_tokens, dtype=np.int64)
        fill = 0

        for start in range(0, df.height, self.chunk_rows):
            chunk = df.slice(start, self.chunk_rows)
            tokens = chunk.explode("token_id").get_column("token_id").to_numpy()

            pos = 0
            while pos < len(tokens):
                need = self.batch_num_tokens - fill
                take = min(need, len(tokens) - pos)
                buf[fill:fill + take] = tokens[pos:pos + take]
                fill += take
                pos += take

                if fill == self.batch_num_tokens:
                    t = torch.as_tensor(buf.copy(), dtype=torch.long)
                    if self.pin_memory and self.device != "cpu":
                        t = t.pin_memory()
                    t = t.to(self.device, non_blocking=self.pin_memory and self.device != "cpu")

                    inputs = t[:-1].view(self.batch_size, self.seq_len)
                    targets = t[1:].view(self.batch_size, self.seq_len)
                    yield TrainingBatch(inputs, targets, size=inputs.numel())
                    fill = 0

In [13]:
dataloader = TrainingDataset(train_histories, batch_size=32, seq_len=100, shuffle=True, device='cuda')

assert len(dataloader) == 1128
batch = next(iter(dataloader))
assert batch.inputs.shape == batch.targets.shape == (32, 100)
assert batch.size == 100 * 32

### 6. Трансформер (энкодер GPT-style)

Здесь вы реализуете **минимальный GPT-энкодер** для последовательностей токенов.

#### Что должно получиться

* Вход: `token_ids` размера `(B, T)`
* Выход энкодера: `x` размера `(B, T, d_model)`
* (Дальше отдельно можно повесить `head` и получить logits `(B, T, vocab_size)`.)


#### 1) `CausalSelfAttention` (у вас уже готово)

Ключевое:

* `qkv = Linear(d_model → 3*d_model)`
* reshape в multi-head
* `F.scaled_dot_product_attention(..., attn_mask=None, is_causal=True)`
* `proj = Linear(d_model → d_model)`

#### 2) `MLP`

Должно быть:

* `Linear(d_model → 4*d_model)`
* `gelu`
* `dropout`
* `Linear(4*d_model → d_model)`


#### 3) `Block` (Pre-LN)

Формула:

* `x = x + dropout(attn(LN(x)))`
* `x = x + dropout(mlp(LN(x)))`

(Один `Dropout` можно переиспользовать.)


#### 4) `GPT` (эмбеддинги + блоки + финальный LN)

* `tok_emb = nn.Embedding(vocab_size, d_model)`
* `pos_emb = nn.Embedding(max_seq_len, d_model)`
* `x = tok_emb(token_ids) + pos_emb(positions)`
* dropout → `n_layers` блоков → `ln_f`

Важно: позиции делаем `0..T-1`, и **T не должен превышать max_seq_len**.


In [14]:
import torch
import torch.nn.functional as F
from torch import nn, Tensor

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.dropout = dropout

    def forward(self, x: Tensor) -> Tensor:
        B, T, D = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, D)
        y = self.proj(y)
        return y


class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model=d_model, dropout=dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


class GPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model=d_model, n_heads=n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, token_ids: Tensor) -> Tensor:
        B, T = token_ids.shape

        pos = torch.arange(T, device=token_ids.device)
        x = self.tok_emb(token_ids) + self.pos_emb(pos)[None, :, :]
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)
            
        x = self.ln_f(x)
        return x

### 7. Граф вычислений (готовая реализация)
Оборачиваем трансформер, голову и подсчет лосса в единый граф вычислений:

In [15]:
class Graph(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        n_layers: int = 4,
        d_model: int = 256,
        n_heads: int = 4,
        dropout: float = 0.0
    ):
        super().__init__()
        self.gpt = GPT(
            vocab_size=vocab_size,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, batch: TrainingBatch):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            x = self.gpt(batch.inputs)
            logits = self.head(x)

        return F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            batch.targets.view(-1)
        )

In [16]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        emb_dim=128,
        max_seq_len=256,
        num_layers=2,
        num_heads=4,
        dropout=0.0
    ):
        super().__init__()
        self.item_embeddings = nn.Embedding(num_items + 1,emb_dim,)
        self.position_embeddings = nn.Embedding(max_seq_len, emb_dim)

        self.layernorm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=4 * emb_dim,
            dropout=dropout,
            activation=nn.GELU(),
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self._init_weights()
    
    @torch.no_grad()
    def _init_weights(self, initializer_range = 0.02):
        for key, value in self.named_parameters():
            if 'weight' in key:
                if 'norm' in key:
                    nn.init.ones_(value.data)
                else:
                    nn.init.trunc_normal_(
                        value.data, std=initializer_range, a=-2 * initializer_range, b=2 * initializer_range
                    )
            else:
                assert 'bias' in key
                nn.init.zeros_(value.data)

    def last_pooling(self, embeddings, mask):
        mask = mask.to(torch.bool)
        flatten_embeddings = embeddings[mask]
        lengths = torch.sum(mask, dim=-1)
        offsets = torch.cumsum(lengths, dim=0) 
        last_embeddings = flatten_embeddings[offsets.long() - 1]
        return last_embeddings

    def forward(self, inputs, attn_mask):
        embeds = self.item_embeddings(inputs)
        bs, num_embs, embs_dim = embeds.shape
        
        positions = torch.arange(start=0, end=num_embs, step=1, device=embeds.device)
        pos_embeds = self.position_embeddings(positions)
        embeds = embeds + pos_embeds
        embeds = self.dropout(self.layernorm(embeds))
        
        mask = torch.triu(torch.ones(num_embs, num_embs, device=embeds.device, dtype=torch.bool), diagonal=1)
        embeds = self.encoder(embeds, mask=mask)
        return embeds, self.last_pooling(embeds, attn_mask)

In [17]:
import torch
import torch.nn.functional as F
from torch import nn


class Graph(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_seq_len: int,
        n_layers: int = 2,
        d_model: int = 128,
        n_heads: int = 4,
        dropout: float = 0.0,
        temperature: float = 1.0,
        normalize: bool = False,
        item_sampling_probs: torch.Tensor | None = None,
        logq_eps: float = 1e-12,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.temperature = temperature
        self.normalize = normalize
        self.logq_eps = logq_eps

        self.sasrec = SASRec(
            num_items=vocab_size - 1,
            emb_dim=d_model,
            max_seq_len=max_seq_len,
            num_layers=n_layers,
            num_heads=n_heads,
            dropout=dropout,
        )

        if item_sampling_probs is None:
            self.register_buffer("item_sampling_probs", torch.empty(0, dtype=torch.float32))
        else:
            probs = torch.as_tensor(item_sampling_probs, dtype=torch.float32)
            if probs.numel() != vocab_size:
                raise ValueError("item_sampling_probs должен иметь длину vocab_size")
            self.register_buffer("item_sampling_probs", probs)

    @staticmethod
    def _get(batch, key, default=None):
        if isinstance(batch, dict):
            return batch.get(key, default)
        return getattr(batch, key, default)

    @staticmethod
    def lengths_to_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
        pos = torch.arange(max_len, device=lengths.device)
        return pos.unsqueeze(0) < lengths.unsqueeze(1)

    def _get_logq(
        self,
        unique_pos_ids: torch.Tensor,
        counts: torch.Tensor,
    ) -> torch.Tensor:
        batch_q = counts.float()
        batch_q = batch_q / batch_q.sum().clamp_min(1.0)

        if self.item_sampling_probs.numel() == self.vocab_size:
            global_q = self.item_sampling_probs[unique_pos_ids].to(batch_q.device)

            q = torch.where(global_q > 0, global_q, batch_q)
        else:
            q = batch_q

        q = q.clamp_min(self.logq_eps)
        return q.log()

    def forward(self, batch) -> torch.Tensor:
        inputs = self._get(batch, "inputs")
        labels = self._get(batch, "labels")
        if labels is None:
            labels = self._get(batch, "targets")

        mask = self._get(batch, "mask")
        if mask is None:
            mask = labels != 0
        mask = mask.to(torch.bool)

        attn_mask = torch.ones_like(inputs, dtype=torch.bool, device=inputs.device)
        seq_embeds, _ = self.sasrec(inputs, attn_mask)

        seq_embeds = seq_embeds.reshape(-1, seq_embeds.size(-1))
        pos_ids = labels.reshape(-1)
        valid = mask.reshape(-1) & (pos_ids != 0)

        seq_embeds = seq_embeds[valid]
        pos_ids = pos_ids[valid]

        if seq_embeds.size(0) == 0:
            return inputs.new_tensor(0.0, dtype=torch.float32, requires_grad=True)

        # Уникальные positive item'ы батча = классы softmax
        unique_pos_ids, target_idx, counts = torch.unique(
            pos_ids,
            sorted=False,
            return_inverse=True,
            return_counts=True,
        )  # [M], [N], [M]

        item_embeds = self.sasrec.item_embeddings(unique_pos_ids)

        if self.normalize:
            seq_embeds = F.normalize(seq_embeds, dim=-1)
            item_embeds = F.normalize(item_embeds, dim=-1)

        logits = (seq_embeds @ item_embeds.T) / self.temperature

        log_q = self._get_logq(unique_pos_ids, counts)
        logits = logits - log_q.unsqueeze(0)

        loss = F.cross_entropy(logits, target_idx)
        return loss

    @torch.no_grad()
    def predict_logits(
        self,
        token_ids: torch.Tensor,
        lengths: torch.Tensor | None = None,
        attn_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        if attn_mask is None:
            if lengths is not None:
                attn_mask = self.lengths_to_mask(lengths, token_ids.size(1))
            else:
                attn_mask = token_ids != 0

        _, pooled = self.sasrec(token_ids, attn_mask)

        item_matrix = self.sasrec.item_embeddings.weight
        if self.normalize:
            pooled = F.normalize(pooled, dim=-1)
            item_matrix = F.normalize(item_matrix, dim=-1)

        logits = (pooled @ item_matrix.T) / self.temperature
        logits[:, 0] = -torch.inf
        return logits

### 8. Цикл обучения (готовая реализация)

Это наш основной механизм обучения - он гоняет батчи, считает лосс, делает `backward()`, обновляет веса, иногда валидируется и всё логирует.

#### Главные сущности

* `graph` — модель (у вас она умеет: `loss = graph(batch)`)
* `train_dataloader` — источник батчей `TrainingBatch(inputs, targets, size)`
* `optimizer` — обновляет параметры модели
* `scheduler` — меняет learning rate по шагам (опционально)
* `validation_func()` — считает метрики на валидации (опционально)
* `SummaryWriter` — пишет логи для TensorBoard

#### Что происходит по шагам

1. **Эпохи и батчи**

* внешний цикл `for epoch in range(num_epochs)`
* внутри идём по батчам `range(len(train_dataloader))`

2. **Валидация раз в `eval_every` шагов**

* если `validation_func` задан и `eval_every != -1`
* запускается при `global_step % eval_every == 0` или на самом последнем шаге
* делаем `graph.eval()` + `torch.inference_mode()` → считаем метрики → обратно `graph.train()`
* пишем метрики в tensorboard: `valid/<name>`

3. **Один train-step**

* `optimizer.zero_grad(set_to_none=True)` — обнуляем градиенты
* `grad_accum_steps` раз:

  * берём батч `batch = next(train_it)`
  * считаем `loss = graph(batch)` под mixed precision (`autocast(bfloat16)`)
  * делим лосс на `grad_accum_steps` (чтобы суммарный градиент был как от большого батча)
  * `loss.backward()`

4. **Логируем train loss**

* `writer.add_scalar("train/loss", train_loss, tokens_passed)`

5. **Gradient clipping (опционально)**

* если `grad_clip > 0`: `clip_grad_norm_`
* логируем `optim/grad_norm`

6. **Шаг оптимизатора + scheduler**

* `optimizer.step()`
* если есть `scheduler`: `scheduler.step()` и логируем `optim/lr`

7. **Скорость**

* меряем время шага
* логируем:

  * `time/train_step_time(s)`
  * `time/tokens_per_sec(k)` (тысяч токенов/сек)


#### Важные детали, на которые стоит обратить внимание

* `tokens_passed` — общий счётчик токенов (ось X в TensorBoard); удобно сравнивать ран по “сколько данных увидели”, а не по “сколько шагов”.
* `curr_tokens` — сколько токенов обработали именно на этом шаге (с учётом grad accumulation).
* `torch.cuda.synchronize()` нужен, чтобы честно мерять время (GPU асинхронный).
* `autocast("cuda", torch.bfloat16)` — ускоряет и экономит память (если модель/железо поддерживает).


In [18]:
import time
import tqdm
import torch
from torch.utils.tensorboard import SummaryWriter

def train_loop(
    graph,
    train_dataloader,
    log_dir: str,
    optimizer,
    scheduler=None,
    num_epochs: int = 1,
    grad_accum_steps: int = 1,
    grad_clip: float = 0.0,
    eval_every: int = -1,
    validation_func=None,
):
    graph.train()
    writer = SummaryWriter(log_dir=log_dir)

    tokens_passed = 0
    global_step = 0

    for epoch in range(num_epochs):
        train_it = iter(train_dataloader)

        num_batches = len(train_dataloader)

        for batch_idx in tqdm.tqdm(range(num_batches), total=num_batches, desc=f"epoch {epoch}"):

            last_step = (epoch == num_epochs - 1) and (batch_idx == num_batches - 1)

            if validation_func is not None and eval_every != -1:
                if global_step % eval_every == 0 or last_step:
                    graph.eval()
                    with torch.inference_mode():
                        metrics = validation_func()
                    graph.train()

                    for name, value in metrics.items():
                        if torch.is_tensor(value):
                            value = float(value.detach().cpu())
                        writer.add_scalar(f"valid/{name}", value, tokens_passed)

            torch.cuda.synchronize()
            train_step_t0 = time.time()

            optimizer.zero_grad(set_to_none=True)

            train_loss = 0.0
            curr_tokens = 0

            for _ in range(grad_accum_steps):
                batch = next(train_it)
                curr_tokens += batch.size

                with torch.autocast("cuda", torch.bfloat16):
                    loss = graph(batch)

                loss = loss / grad_accum_steps
                train_loss += float(loss.detach())
                loss.backward()

            tokens_passed += curr_tokens

            writer.add_scalar("train/loss", train_loss, tokens_passed)

            if grad_clip > 0.0:
                grad_norm = torch.nn.utils.clip_grad_norm_(graph.parameters(), grad_clip)
                writer.add_scalar("optim/grad_norm", float(grad_norm.detach().cpu()), tokens_passed)

            optimizer.step()
            if scheduler is not None:
                scheduler.step()
                writer.add_scalar("optim/lr", optimizer.param_groups[0]["lr"], tokens_passed)

            torch.cuda.synchronize()
            train_step_t1 = time.time()

            dt = train_step_t1 - train_step_t0
            writer.add_scalar("time/train_step_time(s)", dt, tokens_passed)
            writer.add_scalar("time/tokens_per_sec(k)", (curr_tokens / dt) // 1000, tokens_passed)

            global_step += 1

    writer.close()

2026-03-18 10:54:05.298421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773831245.664332     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773831245.755598     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773831246.615172     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773831246.615207     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773831246.615209     106 computation_placer.cc:177] computation placer alr

### 10. Запуск обучения

Здесь непосредственно запускаем обучение. На его прогресс можно (и нужно) посмотреть с помощью `tensorboard --logdir logs`

In [20]:
VOCAB_SIZE = item_to_token_id.height + 1

q_stats = (
    data
    .join(item_to_token_id, on="item_id", how="inner")
    .group_by("token_id")
    .agg(pl.len().alias("cnt"))
    .sort("token_id")
)

item_sampling_probs = torch.zeros(VOCAB_SIZE, dtype=torch.float32)

token_ids = torch.tensor(q_stats["token_id"].to_list(), dtype=torch.long)
cnts = torch.tensor(q_stats["cnt"].to_list(), dtype=torch.float32)

item_sampling_probs[token_ids] = cnts
item_sampling_probs[0] = 0.
item_sampling_probs /= item_sampling_probs.sum()

In [22]:
graph = Graph(
    vocab_size=VOCAB_SIZE,
    max_seq_len=100,
    n_layers=2,
    d_model=128,
    n_heads=4,
    dropout=0.1,
    temperature=1.0,
    normalize=False,
    item_sampling_probs=item_sampling_probs,
).cuda()

compiled_graph = torch.compile(graph)

optimizer = torch.optim.AdamW(
    compiled_graph.parameters(), 
    lr=1e-3, 
    weight_decay=1e-4,
    fused=True
)

train_loop(
    compiled_graph, 
    num_epochs=5,
    train_dataloader=dataloader, 
    log_dir='logs/test', 
    optimizer=optimizer, 
    grad_clip=1.0, 
    grad_accum_steps=1
)

epoch 4: 100%|██████████| 1128/1128 [00:08<00:00, 129.01it/s]


### 11. Подготовка данных для оценки качества

Цель: для каждого пользователя из `test` собрать **историю из трейна**, чтобы потом на ней делать предсказание следующего айтема.

#### Что делаем

1. Берём множество тестовых пользователей:

* `test_users = test.select('uid').unique()`

2. Для этих пользователей берём их события из `train_events` (уже токенизированные):

* фильтруем по `uid` через `semi join`
* сортируем по `timestamp`
* для каждого `uid` оставляем последние `MAX_LEN_PER_USER - 2` токенов
  (потому что 1 токен — BOS, и обычно ещё 1 “слот” оставляем под предсказание)

3. Добавляем `BOS=0` в начало каждой истории:

* конкатим `[uid, BOS]` + `[uid, token_id events]`
* группируем в список `token_id` с `maintain_order=True`

4. (опционально) считаем длину истории и сортируем по длине — удобно для дебага.


In [23]:
MAX_LEN_PER_USER = 100  # итоговая длина истории, включая BOS
BOS = 0

UID_DTYPE = train_events.schema["uid"]
TOKEN_DTYPE = train_events.schema["token_id"]

test_users = test.select(pl.col("uid").cast(UID_DTYPE)).unique()

test_user_events = (
    train_events
    .select([
        pl.col("uid").cast(UID_DTYPE),
        pl.col("token_id").cast(TOKEN_DTYPE),
        pl.col("timestamp"),
    ])
    .join(test_users, on="uid", how="semi")
    .sort("timestamp")
    .group_by("uid", maintain_order=True)
    .tail(MAX_LEN_PER_USER - 2)
    .select([
        pl.col("uid").cast(UID_DTYPE),
        pl.col("token_id").cast(TOKEN_DTYPE),
    ])
)

test_histories = (
    pl.concat([
        test_users.select([
            pl.col("uid").cast(UID_DTYPE),
            pl.lit(BOS).cast(TOKEN_DTYPE).alias("token_id"),
        ]),
        test_user_events.select([
            pl.col("uid").cast(UID_DTYPE),
            pl.col("token_id").cast(TOKEN_DTYPE),
        ]),
    ])
    .group_by("uid", maintain_order=True)
    .agg(pl.col("token_id"))
)

test_histories = test_histories.with_columns(length=pl.col("token_id").list.len())
test_histories = test_histories.sort("length", descending=True)

In [24]:
assert test_users.height == test_histories.height == 37446


### 12. Тестовый датасет

* в train мы выдавали непрерывный поток токенов
* в test мы работаем с **отдельными историями пользователей**
* поэтому здесь нужен **padding внутри батча**



In [25]:
from dataclasses import dataclass

import polars as pl
import torch
from torch.nn.utils.rnn import pad_sequence


@dataclass
class TestBatch:
    token_ids: torch.Tensor
    lengths: torch.Tensor


class TestDataset:
    def __init__(
        self,
        df: pl.DataFrame,
        batch_size: int,
        device: str = "cuda",
    ):
        self.df = df
        self.batch_size = batch_size
        self.device = device
        self._tensors = [
            torch.tensor(seq, dtype=torch.long)
            for seq in self.df.get_column("token_id").to_list()
        ]

    def __len__(self):
        return (len(self._tensors) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        for start in range(0, len(self._tensors), self.batch_size):
            batch_tensors = self._tensors[start:start + self.batch_size]
            lengths = torch.tensor([len(x) for x in batch_tensors], dtype=torch.long)
            token_ids = pad_sequence(batch_tensors, batch_first=True, padding_value=0)
            yield TestBatch(
                token_ids=token_ids.to(self.device),
                lengths=lengths.to(self.device),
            )

## 13. Оценка качества

Идея:

1. прогоняем истории через GPT → получаем hidden states
2. берём hidden state **последнего реального токена** (по `lengths-1`)
3. через `head` получаем logits по словарю
4. берём topK токенов-кандидатов, маппим обратно в `item_id`
5. считаем метрики `evaluate(...)`

In [27]:
import torch
import polars as pl

TOPK = 100

ds = TestDataset(test_histories, batch_size=128)
graph = graph.eval()

with torch.inference_mode():
    all_candidates = []

    for batch in ds:
        with torch.autocast("cuda", torch.bfloat16):
            logits = graph.predict_logits(batch.token_ids, batch.lengths)

        _, indices = torch.topk(logits, k=TOPK, dim=-1)
        all_candidates.append(indices.cpu())

candidates = torch.cat(all_candidates, dim=0)

candidates_df = pl.DataFrame({
    "uid": test_histories.get_column("uid"),
    "token_id": candidates.tolist(),
})

candidates_df = candidates_df.explode("token_id")
candidates_df = candidates_df.join(item_to_token_id, on="token_id", how="left")
candidates_df = candidates_df.group_by("uid", maintain_order=True).agg(pl.col("item_id"))

evaluate(
    targets=test_targets,
    candidates=dict(candidates_df.iter_rows()),
    catalog_size=train.get_column("item_id").n_unique(),
    topk=TOPK,
)

{'hitrate': 0.3746995673770229,
 'recall': 0.1314229545358867,
 'ndcg': np.float64(0.051988557177676586),
 'coverage': 0.16244265288851276}